# Activity 7 - Additional Exercises (NumPy and Pandas)
SECOM Semiconductor Manufacturing dataset

Run order: Setup cell first, then Exercises 1-3.

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
from urllib.request import urlretrieve

# ---- Student configuration (same values as Activity 7, Cell 1) ----
LAST_NAME = "Adonis"
STUDENT_ID = "TUPM-26-1828"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
assert id_digits, "Student ID must contain digits."
SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
NAME_LENGTH = len(LAST_NAME)
RANDOM_SEED = int(id_digits) % (2**32)
verification_value = ID_SUM * NAME_LENGTH + SEED_DIGIT
np.set_printoptions(precision=4, suppress=True)

# ---- Paths (finds data/ in this folder or in the parent repo folder) ----
WORK_DIR = Path.cwd()
DATA_DIR = next((p / "data" for p in [WORK_DIR, WORK_DIR.parent] if (p / "data").is_dir()),
                WORK_DIR / "data")
OUTPUT_DIR = WORK_DIR
for p in [DATA_DIR / "secom.data", DATA_DIR / "secom_labels.data"]:
    if not p.is_file():
        matches = list(WORK_DIR.rglob(p.name))
        if matches:
            p = matches[0]
            DATA_DIR = p.parent
        else:
            DATA_DIR.mkdir(parents=True, exist_ok=True)
            url_name = p.name
            url = f"https://archive.ics.uci.edu/ml/machine-learning-databases/secom/{url_name}"
            print(f"Downloading {url_name}...")
            urlretrieve(url, p)
            if not p.is_file():
                raise FileNotFoundError(
                    f"Missing file: {p}\n"
                    f"Place secom.data and secom_labels.data in {DATA_DIR}"
                )

# ---- Load data (original order preserved) ----
raw = np.genfromtxt(DATA_DIR / "secom.data", dtype=float)
labels = np.loadtxt(DATA_DIR / "secom_labels.data", usecols=0, dtype=int)
assert raw.ndim == 2 and len(labels) == len(raw)
assert np.isin(labels, [-1, 1]).all()

n_records, n_features = raw.shape
feature_names = np.array([f"Feature_{i:03d}" for i in range(n_features)])
record_ids = np.arange(n_records)          # original SECOM record identifiers

# ---- Usable features: observed values exist and are not constant ----
missing = np.isnan(raw)
observed_counts = (~missing).sum(axis=0)
obs_min = np.where(missing, np.inf, raw).min(axis=0)
obs_max = np.where(missing, -np.inf, raw).max(axis=0)
constant = (observed_counts > 0) & (obs_min == obs_max)
usable_mask = (observed_counts >= 2) & ~constant
usable_idx = np.flatnonzero(usable_mask)

print("Student:", LAST_NAME, STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Name Length:", NAME_LENGTH)
print("Random Seed:", RANDOM_SEED)
print("Verification Value:", verification_value)
print("Shape:", raw.shape, "| Usable features:", len(usable_idx))


Student: Adonis TUPM-26-1828
Seed Digit: 8
ID Sum: 27
Name Length: 6
Random Seed: 261828
Verification Value: 170
Shape: (1567, 590) | Usable features: 474


## Exercise 1: Vectorized Measurement Quality Analysis

In [2]:
# 1. Features with missing or constant observed values
has_missing = missing.any(axis=0)
print("Features with missing values:", int(has_missing.sum()))
print("Constant features:", int(constant.sum()))

# 2. Reproducibly select 5 + (SEED_DIGIT % 3) usable features
n_select = 5 + (SEED_DIGIT % 3)
rng1 = np.random.default_rng(RANDOM_SEED)
sel_idx = np.sort(rng1.choice(usable_idx, size=n_select, replace=False))
sel_names = feature_names[sel_idx]
X = raw[:, sel_idx]

# 3. NumPy feature-wise statistics
means = np.nanmean(X, axis=0)
medians = np.nanmedian(X, axis=0)
sds = np.nanstd(X, axis=0, ddof=1)
miss_pct = np.isnan(X).mean(axis=0) * 100
observed_n = (~np.isnan(X)).sum(axis=0)

# 4. Robust standardized values (median/MAD), zero MAD handled safely
dev = X - medians
mad = np.nanmedian(np.abs(dev), axis=0)
zero_mad = mad == 0
safe_mad = np.where(zero_mad, 1.0, mad)
robust_z = np.where(np.isnan(X), np.nan, np.where(zero_mad, 0.0, 0.6745 * dev / safe_mad))

# 5. Flag |robust z| > 3.5
flagged = np.abs(robust_z) > 3.5
flag_count = flagged.sum(axis=0)
flag_pct = flag_count / observed_n * 100

# 6. Rank by flagged percentage and export CSV
summary = pd.DataFrame({
    "Feature": sel_names,
    "Mean": means,
    "Median": medians,
    "Sample_SD": sds,
    "Missing_Percentage": miss_pct,
    "MAD": mad,
    "Zero_MAD": zero_mad,
    "Flagged_Count": flag_count,
    "Flagged_Percentage": flag_pct,
}).sort_values(["Flagged_Percentage", "Feature"], ascending=[False, True]).reset_index(drop=True)
summary.insert(0, "Rank", np.arange(1, len(summary) + 1))
summary.round(4).to_csv(OUTPUT_DIR / "measurement_quality.csv", index=False)

# Verify vectorized results against pandas and a plain loop
chk = pd.DataFrame(X, columns=sel_names)
stats_ok = (np.allclose(chk.mean().to_numpy(), means) and
            np.allclose(chk.median().to_numpy(), medians) and
            np.allclose(chk.std(ddof=1).to_numpy(), sds))
loop_total = 0
for j in range(X.shape[1]):
    col = X[:, j][~np.isnan(X[:, j])]
    m = np.median(col)
    d = np.median(np.abs(col - m))
    if d > 0:
        loop_total += int((np.abs(0.6745 * (col - m) / d) > 3.5).sum())
verified = bool(stats_ok and loop_total == int(flagged.sum()))

print(summary.round(4).to_string(index=False))
print()
print("=== ASSESSMENT DATA: EXERCISE 1 ===")
hi_miss = int(np.argmax(miss_pct))
print("Selected Features:", sel_names.tolist())
print(f"Highest Missing Percentage: {miss_pct[hi_miss]:.2f}% ({sel_names[hi_miss]})")
print("Feature with Highest Flagged Percentage:", summary.loc[0, "Feature"],
      f"({summary.loc[0, 'Flagged_Percentage']:.2f}%)")
print("Total Flagged Observations:", int(flagged.sum()))
print("Vectorized Results Verified:", verified)


Features with missing values: 538
Constant features: 116
 Rank     Feature    Mean  Median  Sample_SD  Missing_Percentage    MAD  Zero_MAD  Flagged_Count  Flagged_Percentage
    1 Feature_543  0.0085  0.0078     0.0015              0.1276 0.0001     False            628             40.1278
    2 Feature_025  1.2038  1.2830     0.1776              0.1276 0.0257     False            398             25.4313
    3 Feature_548 75.6799 74.0840     3.3905             16.5922 1.2180     False            217             16.6029
    4 Feature_037 66.2213 66.2318     0.3041              0.0638 0.1456     False             72              4.5977
    5 Feature_238  0.0048  0.0046     0.0017              0.1276 0.0010     False             12              0.7668
    6 Feature_246  4.1487  3.6730    10.0451             64.9649 0.8636     False              3              0.5464
    7 Feature_323  5.3934  5.4588     2.5189              0.0638 1.0041     False              3              0.1916

=== AS

## Exercise 2: Pass/Fail Grouped Profiling (balanced sample, standardized mean differences)

In [3]:
# 1. Labeled DataFrame preserving original record identifiers
df = pd.DataFrame(raw, columns=feature_names, index=pd.Index(record_ids, name="Example_ID"))
df["Result"] = labels
df["Outcome"] = np.where(labels == -1, "Pass", "Fail")

# 2. Reproducible balanced sample: min(20 + ID_SUM, Pass count, Fail count) per outcome
n_pass = int((labels == -1).sum())
n_fail = int((labels == 1).sum())
n_per_group = min(20 + ID_SUM, n_pass, n_fail)
rng2 = np.random.default_rng(RANDOM_SEED)
pass_ids = df.index[df["Outcome"] == "Pass"].to_numpy()
fail_ids = df.index[df["Outcome"] == "Fail"].to_numpy()
chosen = np.sort(np.concatenate([
    rng2.choice(pass_ids, size=n_per_group, replace=False),
    rng2.choice(fail_ids, size=n_per_group, replace=False),
]))
balanced = df.loc[chosen].copy()

# 3. Five usable features + median imputation using FULL-dataset medians
sel2_idx = np.sort(rng2.choice(usable_idx, size=5, replace=False))
sel2 = feature_names[sel2_idx].tolist()
full_medians = pd.Series(np.nanmedian(raw[:, sel2_idx], axis=0), index=sel2)
balanced[sel2] = balanced[sel2].fillna(full_medians)
assert balanced[sel2].isna().sum().sum() == 0

# 4. groupby() + agg()
grouped_stats = balanced.groupby("Outcome")[sel2].agg(["count", "mean", "median", "std"])
print(grouped_stats.round(4).T.to_string())

# 5. Absolute mean difference and standardized mean difference (pooled SD)
g = balanced.groupby("Outcome")[sel2]
mean_g, sd_g, n_g = g.mean(), g.std(ddof=1), g.count()
n_f, n_p = n_g.loc["Fail"], n_g.loc["Pass"]
pooled = np.sqrt(((n_f - 1) * sd_g.loc["Fail"] ** 2 + (n_p - 1) * sd_g.loc["Pass"] ** 2)
                 / (n_f + n_p - 2))
abs_diff = (mean_g.loc["Fail"] - mean_g.loc["Pass"]).abs()
smd = (abs_diff / pooled.where(pooled > 0)).fillna(0.0)   # zero denominators -> 0

# 6. Rank by absolute standardized mean difference and export
comparison2 = pd.DataFrame({
    "Feature": sel2,
    "Fail_Mean": mean_g.loc["Fail"].reindex(sel2).to_numpy(),
    "Pass_Mean": mean_g.loc["Pass"].reindex(sel2).to_numpy(),
    "Abs_Mean_Difference": abs_diff.reindex(sel2).to_numpy(),
    "Pooled_SD": pooled.reindex(sel2).to_numpy(),
    "Abs_Standardized_Mean_Difference": smd.reindex(sel2).to_numpy(),
}).sort_values("Abs_Standardized_Mean_Difference", ascending=False).reset_index(drop=True)
comparison2.insert(0, "Rank", np.arange(1, len(comparison2) + 1))
comparison2.round(4).to_csv(OUTPUT_DIR / "feature_comparison.csv", index=False)
balanced.reset_index().to_csv(OUTPUT_DIR / "balanced_sample.csv", index=False)

# NumPy cross-check
fa = balanced.loc[balanced["Outcome"] == "Fail", sel2].to_numpy()
pa = balanced.loc[balanced["Outcome"] == "Pass", sel2].to_numpy()
np_abs = np.abs(fa.mean(axis=0) - pa.mean(axis=0))
np_pooled = np.sqrt(((len(fa) - 1) * fa.var(axis=0, ddof=1) + (len(pa) - 1) * pa.var(axis=0, ddof=1))
                    / (len(fa) + len(pa) - 2))
np_smd = np.divide(np_abs, np_pooled, out=np.zeros_like(np_abs), where=np_pooled > 0)
match = bool(np.allclose(np_smd, smd.reindex(sel2).to_numpy(), rtol=1e-10, atol=1e-10))

print()
print(comparison2.round(4).to_string(index=False))
print()
print("=== ASSESSMENT DATA: EXERCISE 2 ===")
print("Balanced Sample Size:", len(balanced), f"({n_per_group} per outcome)")
print("Selected Features:", sel2)
print("Highest-Ranked Feature:", comparison2.loc[0, "Feature"])
print(f"Absolute Standardized Mean Difference: {comparison2.loc[0, 'Abs_Standardized_Mean_Difference']:.4f}")
print("NumPy-Pandas Statistics Match:", match)


Outcome                 Fail      Pass
Feature_018 count    48.0000   48.0000
            mean    189.9388  190.0859
            median  189.9044  189.7337
            std       2.7702    2.8214
Feature_150 count    48.0000   48.0000
            mean      7.1284    7.4510
            median    6.0170    6.2315
            std       3.6387    3.2559
Feature_272 count    48.0000   48.0000
            mean     45.2876   42.5731
            median   46.2616   41.7155
            std      15.0347   17.0598
Feature_310 count    48.0000   48.0000
            mean      0.3053    0.3056
            median    0.3067    0.2964
            std       0.0329    0.0443
Feature_429 count    48.0000   48.0000
            mean      3.7165    4.3551
            median    3.1886    4.1198
            std       2.1036    1.7811

 Rank     Feature  Fail_Mean  Pass_Mean  Abs_Mean_Difference  Pooled_SD  Abs_Standardized_Mean_Difference
    1 Feature_429     3.7165     4.3551               0.6386     1.9490   

## Exercise 3: Statistical Robustness and Sensitivity Analysis

In [4]:
pass_mask = labels == -1
fail_mask = labels == 1

# 1. Five usable features with >= 10 observed values in each outcome group
obs_pass = (~np.isnan(raw[pass_mask])).sum(axis=0)
obs_fail = (~np.isnan(raw[fail_mask])).sum(axis=0)
eligible = np.flatnonzero(usable_mask & (obs_pass >= 10) & (obs_fail >= 10))
sel3 = np.sort(np.random.default_rng(RANDOM_SEED).choice(eligible, size=5, replace=False))
sel3_names = feature_names[sel3]
BOOT_REPS = 200 + 10 * SEED_DIGIT
STRATEGIES = ["available_case", "median_imputation"]

# 2. Two preparation strategies
def prepare(j, strategy):
    col = raw[:, j]
    if strategy == "available_case":
        p, f = col[pass_mask], col[fail_mask]
        return p[~np.isnan(p)], f[~np.isnan(f)]
    filled = np.where(np.isnan(col), np.nanmedian(col), col)   # full-dataset median
    return filled[pass_mask], filled[fail_mask]

def pooled_sd(p, f):
    if len(p) < 2 or len(f) < 2:
        return 0.0
    v = ((len(p) - 1) * p.var(ddof=1) + (len(f) - 1) * f.var(ddof=1)) / (len(p) + len(f) - 2)
    return float(np.sqrt(v))

# 3-5. Mean difference (Fail - Pass) + stratified bootstrap with percentile 95% CI
def run_analysis(seed):
    rng = np.random.default_rng(seed)
    rows = []
    for j, name in zip(sel3, sel3_names):
        for strategy in STRATEGIES:
            p, f = prepare(j, strategy)
            diff = f.mean() - p.mean()
            sp = pooled_sd(p, f)
            smd = abs(diff) / sp if sp > 0 else 0.0
            p_idx = rng.integers(0, len(p), size=(BOOT_REPS, len(p)))   # resample within Pass
            f_idx = rng.integers(0, len(f), size=(BOOT_REPS, len(f)))   # resample within Fail
            boot = f[f_idx].mean(axis=1) - p[p_idx].mean(axis=1)
            lo, hi = np.percentile(boot, [2.5, 97.5])
            rows.append({"Feature": name, "Strategy": strategy, "Pass_n": len(p), "Fail_n": len(f),
                         "Mean_Diff": diff, "Abs_Std_Mean_Diff": smd,
                         "Boot_Mean_Diff": boot.mean(), "CI_Lower": lo, "CI_Upper": hi})
    return pd.DataFrame(rows)

results = run_analysis(RANDOM_SEED)

# 6. Compare strategies (direction, CIs, rankings)
ac = results[results["Strategy"] == "available_case"].set_index("Feature").loc[sel3_names]
mi = results[results["Strategy"] == "median_imputation"].set_index("Feature").loc[sel3_names]
comp = pd.DataFrame({
    "AC_Mean_Diff": ac["Mean_Diff"], "MI_Mean_Diff": mi["Mean_Diff"],
    "Change_In_Mean_Diff": mi["Mean_Diff"] - ac["Mean_Diff"],
    "AC_CI_Lower": ac["CI_Lower"], "AC_CI_Upper": ac["CI_Upper"],
    "MI_CI_Lower": mi["CI_Lower"], "MI_CI_Upper": mi["CI_Upper"],
    "AC_CI_Excludes_Zero": (ac["CI_Lower"] > 0) | (ac["CI_Upper"] < 0),
    "MI_CI_Excludes_Zero": (mi["CI_Lower"] > 0) | (mi["CI_Upper"] < 0),
    "AC_Abs_SMD": ac["Abs_Std_Mean_Diff"], "MI_Abs_SMD": mi["Abs_Std_Mean_Diff"],
})
comp["Sign_Changed"] = np.sign(comp["AC_Mean_Diff"]) != np.sign(comp["MI_Mean_Diff"])
comp["AC_Rank"] = comp["AC_Abs_SMD"].rank(ascending=False, method="first").astype(int)
comp["MI_Rank"] = comp["MI_Abs_SMD"].rank(ascending=False, method="first").astype(int)
comp["Rank_Changed"] = comp["AC_Rank"] != comp["MI_Rank"]

# 7. Reproducibility check, then export
rerun = run_analysis(RANDOM_SEED)
try:
    pd.testing.assert_frame_equal(results, rerun)
    reproducible = True
except AssertionError:
    reproducible = False

comp.round(4).reset_index().to_csv(OUTPUT_DIR / "robustness_comparison.csv", index=False)

largest_feat = comp["Change_In_Mean_Diff"].abs().idxmax()
largest_chg = abs(comp.loc[largest_feat, "Change_In_Mean_Diff"])
n_sign = int(comp["Sign_Changed"].sum())
n_rank = int(comp["Rank_Changed"].sum())

report = (
    "SENSITIVITY REPORT\n"
    f"Student: {LAST_NAME}\n"
    f"Random Seed: {RANDOM_SEED}\n"
    f"Selected Features: {', '.join(sel3_names)}\n"
    f"Bootstrap Repetitions: {BOOT_REPS}\n"
    f"Largest Change in Mean Difference: {largest_chg:.4f} ({largest_feat})\n"
    f"Features with Changed Difference Sign: {n_sign}\n"
    f"Features with Changed Ranking: {n_rank}\n"
    f"Reproducibility Verified: {reproducible}\n\n"
    "Interpretation:\n"
    "- Mean differences (Fail - Pass) were compared between available-case analysis and\n"
    "  median imputation using the full dataset.\n"
    "- Median imputation can shrink variability, so confidence intervals and rankings may shift.\n"
    "- Differences are descriptive and do not establish causes of manufacturing failure.\n"
)
(OUTPUT_DIR / "sensitivity_report.txt").write_text(report, encoding="utf-8")

print(comp.round(4).to_string())
print()
print("=== ASSESSMENT DATA: EXERCISE 3 ===")
print("Selected Features:", sel3_names.tolist())
print("Bootstrap Repetitions:", BOOT_REPS)
print(f"Largest Change in Mean Difference: {largest_chg:.4f} ({largest_feat})")
print("Features with Changed Difference Sign:", n_sign)
print("Features with Changed Ranking:", n_rank)
print("Reproducibility Verified:", reproducible)


             AC_Mean_Diff  MI_Mean_Diff  Change_In_Mean_Diff  AC_CI_Lower  AC_CI_Upper  MI_CI_Lower  MI_CI_Upper  AC_CI_Excludes_Zero  MI_CI_Excludes_Zero  AC_Abs_SMD  MI_Abs_SMD  Sign_Changed  AC_Rank  MI_Rank  Rank_Changed
Feature                                                                                                                                                                                                                         
Feature_025       -0.0262       -0.0252               0.0009      -0.0672       0.0212      -0.0827       0.0196                False                False      0.1474      0.1421         False        3        3         False
Feature_037        0.0505        0.0505              -0.0000      -0.0032       0.1062      -0.0054       0.1069                False                False      0.1660      0.1660         False        2        2         False
Feature_239       -0.0001       -0.0001               0.0000      -0.0004       0.0002      -0.0004 